In [0]:
# Databricks notebook source
# =============================================================================
# monitoring_tool_orchestrator_final.py
# =============================================================================
# Batch file-monitoring orchestrator.
# One usecase per run (widget-driven).
#
# Architecture
# ------------
#   Config  → Delta Table monitoring.usecase_config  (one row per file)
#   Utils   → monitoring_utils_v3_final.py
#   Alarms  → Delta Table monitoring.alarm_log
#
# What changed from the previous version
# ----------------------------------------
# [O1] load_usecase_config now returns a DataFrame → replaced every
#      config["key"] access with meta["key"] (from get_scalar_config).
# [O2] Removed manual frequency/lag_days groupBy validation block —
#      get_scalar_config raises if the DF is empty; the window helper
#      raises if frequency is invalid.
# [O3] Removed intermediate data_files concat cell (was unused by
#      get_expected_files_df which reads file_template directly).
# [O4] Step 4: get_expected_files_df returns a DataFrame with
#      resolved_filename column — no more Python list of dicts.
# [O5] Step 5: list_raw_files_df returns a DataFrame of found files —
#      replaces old list_raw_files(base_path, data_files) call.
# [O6] Step 6: missing-file detection is a single DataFrame anti-join —
#      replaces the Python list comprehension over found_files set.
# [O7] Step 8: write_alarm_records_df writes all alarms in one Spark
#      action — replaces the per-file for-loop.


In [ ]:
import tomllib

with open("pyproject.toml", "rb") as f: # Le fichier doit être ouvert en mode binaire
    dictionnaire_config = tomllib.load(f)
    path_notebook_utils=dictionnaire_config.get("paths")["utils_file"]

In [0]:
%run $path_notebook_utils

In [0]:

import pytz
from datetime import datetime, timedelta
from pyspark.sql import functions as F

# COMMAND ----------
# ---------------------------------------------------------------------------
#  1. WIDGETS — which usecase and which date to monitor
# ---------------------------------------------------------------------------

dbutils.widgets.text("usecase_name", "erogazione_offerte", "Use Case Name")
usecase_name = dbutils.widgets.get("usecase_name").strip().lower().replace(" ", "_")

dbutils.widgets.text("DATE", "today", "yyyy-MM-dd  or  'today'")
date_input = dbutils.widgets.get("DATE").strip()

print(f"usecase_name : {usecase_name}")
print(f"date_input   : {date_input}")

In [ ]:

# COMMAND ----------
# ---------------------------------------------------------------------------
#  Parse sysdate
# ---------------------------------------------------------------------------

if date_input == "today":
    sysdate = datetime.now(pytz.utc)
else:
    sysdate = datetime.strptime(date_input, "%Y-%m-%d").replace(tzinfo=pytz.utc)

print(f"sysdate : {sysdate}")

# COMMAND ----------
# ---------------------------------------------------------------------------
#  2. LOAD CONFIG  →  Spark DataFrame, one row per expected file
# ---------------------------------------------------------------------------

config_df = load_usecase_config(usecase_name)

print(f"✅  Config loaded for: {usecase_name}")
display(config_df)


In [0]:

# COMMAND ----------
# ---------------------------------------------------------------------------
#  Extract usecase-level scalar values once.
#  All downstream helpers that need a single scalar (frequency, lag_days, …)
#  read from `meta` — no more `config["key"]` on an undefined dict.
# ---------------------------------------------------------------------------

meta = get_scalar_config(config_df)

print("Scalar config:")
for k, v in meta.items():
    print(f"  {k:20s} = {v}")

# COMMAND ----------
# ---------------------------------------------------------------------------
#  3. CHECK WHETHER THE INSPECTION WINDOW HAS BEEN REACHED
# ---------------------------------------------------------------------------

window_start, window_end = get_inspection_window(
    frequency = meta["frequency"],
    lag_days  = meta["lag_days"],
    sysdate   = sysdate,
)

print(f"\n🔍  Inspection window: {window_start}  →  {window_end}")

if sysdate < window_start:
    print("⏳  Window not yet reached. Nothing to check.")
    dbutils.notebook.exit("WINDOW_NOT_REACHED")


In [0]:

# COMMAND ----------
# ---------------------------------------------------------------------------
#  4. BUILD EXPECTED FILENAMES
#     get_expected_files_df adds a `resolved_filename` column to config_df
#     via a Spark UDF — no collect() on the file list.
# ---------------------------------------------------------------------------

expected_df = get_expected_files_df(config_df, sysdate)

print("Expected files:")
display(expected_df.select("subdirectory", "file_template", "resolved_filename", "raw_data_path"))


In [0]:

# COMMAND ----------
# ---------------------------------------------------------------------------
#  5. SCAN RAW-DATA STORAGE
#     list_raw_files_df collects distinct (raw_data_path, subdirectory) pairs
#     on the driver, calls dbutils.fs.ls for each, then returns a DataFrame.
# ---------------------------------------------------------------------------

found_df = list_raw_files_df(config_df)

print(f"📂  Files found on lake: {found_df.count()}")
# display(found_df)




In [0]:
# COMMAND ----------
# ---------------------------------------------------------------------------
#  6. DETECT MISSING FILES  —  DataFrame anti-join
#     A file is missing when (raw_data_path, subdirectory, resolved_filename)
#     has no match in (raw_data_path, subdirectory, found_filename).
# ---------------------------------------------------------------------------

missing_df = expected_df.join(
    found_df,
    on=(
        (expected_df.raw_data_path    == found_df.raw_data_path)  &
        (expected_df.subdirectory     == found_df.subdirectory)   &
        (expected_df.resolved_filename == found_df.found_filename)
    ),
    how="left_anti",
)

missing_count = missing_df.count()
if missing_count == 0:
    print("\n✅  No alarm — all expected files are present.")
    dbutils.notebook.exit("OK")

print(f"\n⚠️   Missing files: {missing_count}")
display(missing_df.select("subdirectory", "resolved_filename", "raw_data_path"))

In [0]:

# COMMAND ----------
# ---------------------------------------------------------------------------
#  7. QUERY ADF + COMPUTE ALARM LEVEL
# ---------------------------------------------------------------------------

last_run = retrieve_last_adf_run(
    pipeline_name = meta["pipeline_name"],
    freq          = meta["frequency"],
    sysdate       = sysdate,
)

alarm_level = compute_alarm_level(
    sysdate       = sysdate,
    last_run      = last_run,
    frequency     = meta["frequency"],
    deadline_rule = meta["deadline_rule"],
    lag_days      = meta["lag_days"],
)

label = ALARM_LEVELS[alarm_level]
print(f"\n🚨  ALARM LEVEL {alarm_level}:  {label}")


In [0]:

# COMMAND ----------
# ---------------------------------------------------------------------------
#  8. WRITE ALARM RECORDS  —  one Spark action, not a per-file loop
# ---------------------------------------------------------------------------

n_written = write_alarm_records_df(
    missing_df       = missing_df,
    check_date       = sysdate,
    alarm_level      = alarm_level,
    alarm_label      = label,
    usecase_name     = usecase_name,
    input_source     = meta.get("sender_type") or "",
    usecase_deadline = meta.get("deadline_rule"),
)

print(f"\n📝  {n_written} alarm record(s) written to `monitoring.alarm_log`.")


In [0]:
from pyspark.sql import functions as F
from datetime import datetime
log_table =(
spark.table("monitoring.alarm_log")\
    .withColumn(
        "cuurent_date",
        F.current_date(),
    )
    .select("cuurent_date","usecase","check_date","alarm_level","alarm_label","missing_file","usecase_deadline")
    .filter(F.col("cuurent_date") == F.current_date())
    .orderBy(F.col("check_date").desc())
    .distinct()

)

In [ ]:
from pyspark.sql import functions as F
from datetime import datetime
log_table =(
spark.table("monitoring.alarm_log")\
    .withColumn(
        "cuurent_date",
        F.current_date(),
    )
    .select("cuurent_date","usecase","check_date","alarm_level","alarm_label","missing_file","usecase_deadline")
    .filter(F.col("cuurent_date") == F.current_date())
    .orderBy(F.col("check_date").desc())
    .distinct()

)

%python

!pip install exchangelib
from exchangelib import DELEGATE,Account, Credentials, Configuration, Message, FileAttachment , HTMLBody
import datetime as dt
from dateutil.relativedelta import relativedelta

recipients_list=[]
cc_list=[]

pdf = log_table.toPandas()

html_table = pdf.to_html(index=False)

try:
      creds = Credentials(username='YOUR_USERNAME', password='YOUR_PASSWORD')
      config = Configuration(service_endpoint='REPLACE_BY_YOUR_ENDPOINY', credentials=creds)
      account = Account(primary_smtp_address='STMP_EMAIL', config=config, autodiscover=False, access_type=DELEGATE)
      pass
except:
    pass


message = Message(
    account=account,
    subject="Alarm Log Report",
    body=HTMLBody(f"""
        <style>
            table {{
                border-collapse: collapse;
                width: 100%;
                font-size: 11px;
                font-family: Arial, sans-serif;
            }}
            th {{
                background-color: #1e3a8a;
                color: white;
                padding: 8px;
                text-align: center;
                font-size: 10px;
                font-weight: bold;
                text-transform: uppercase;
            }}
            td {{
                padding: 6px 8px;
                text-align: center;
                border-bottom: 1px solid #ddd;
                font-size: 11px;
            }}
            tr:nth-child(even) {{
                background-color: #f8fafc;
            }}
            tr:nth-child(odd) {{
                background-color: #ffffff;
            }}
            .badge-red {{
                display: inline-block;
                padding: 2px 8px;
                border-radius: 10px;
                font-size: 10px;
                font-weight: bold;
                background-color: #fee2e2;
                color: #991b1b;
            }}
            .badge-orange {{
                display: inline-block;
                padding: 2px 8px;
                border-radius: 10px;
                font-size: 10px;
                font-weight: bold;
                background-color: #ffedd5;
                color: #9a3412;
            }}
            .file-name {{
                font-family: Consolas, 'Courier New', monospace;
                font-size: 10px;
                color: #374151;
                word-break: break-all;
            }}
            .none-text {{
                color: #9ca3af;
                font-style: italic;
                font-size: 10px;
            }}
            h3 {{
                text-align: center;
                color: white;
                background-color: #1e3a8a;
                padding: 12px;
                margin: 0 0 10px 0;
                font-size: 14px;
                border-radius: 4px 4px 0 0;
            }}
        </style>
        
        <h3>🔔 Alarm Log Report</h3>
        {html_table}
    """),
   to_recipients=recipients_list,
   cc_recipients=cc_list

)





message.send()